# 05 - Agent Evaluation

## Scenario: Benchmarking the Northstar Triage Agent

How do you know your agent is actually getting better? If you tweak the system prompt, did you fix the refund bug but break the password reset flow?

You cannot evaluate agents by "vibes". You must build an **Evaluation Harness**.

In this notebook, we will:
1. Define a golden dataset of test cases.
2. Build an **LLM-as-a-Judge** using OpenAI Structured Outputs to programmatically grade our agent's accuracy and tool choices.

In [1]:
import os
from openai import OpenAI
from pydantic import BaseModel, Field

# 1. Attempt to use the real API
if os.environ.get("OPENAI_API_KEY"):
    client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))
else:
    # 2. Fallback to our local mock for students without keys
    print("⚠️ No OPENAI_API_KEY found. Falling back to MockOpenAI...")
    import sys
    import os
    sys.path.append(os.path.abspath("../../.."))
    from awsome_agents.mock_openai import MockOpenAI
    client = MockOpenAI()

# 3. Optional: Local LLMs
# If you prefer to use a local model like Llama 3 instead of the mock:
# client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")

# 1. Our Golden Dataset
test_cases = [
    {"input": "My account is locked.", "expected_category": "Auth", "expected_tool": "check_user"},
    {"input": "I need a $50 refund for my pro plan.", "expected_category": "Billing", "expected_tool": "prepare_refund"},
    {"input": "The EU server is down.", "expected_category": "Engineering", "expected_tool": "check_server"}
]


## 1. The Agent (The System Under Test)

In [2]:
# We simulate our agent's output for the sake of the tutorial
def mock_run_agent(ticket: str) -> dict:
    if "locked" in ticket:
        return {"category": "Auth", "tool_used": "check_user", "response": "I checked your account."}
    elif "refund" in ticket:
        return {"category": "Billing", "tool_used": "escalate", "response": "I escalated this to billing."} # Purposeful failure!
    else:
        return {"category": "Engineering", "tool_used": "check_server", "response": "EU server is failing."}


## 2. LLM-as-a-Judge

We can write python `assert` statements for exact matches, but sometimes we need to grade the *quality* of the text. We use a secondary LLM with strict Pydantic output to grade the primary agent.

In [3]:
class AgentGrade(BaseModel):
    is_correct: bool = Field(description="True if the agent handled the ticket perfectly.")
    reason: str = Field(description="Explanation for the grade.")

def grade_agent_response(ticket: str, agent_output: dict, expected_tool: str) -> AgentGrade:
    print(f"⚖️ Grading ticket: '{ticket}'")
    
    # Simple deterministic grade for tools
    if agent_output["tool_used"] != expected_tool:
        return AgentGrade(is_correct=False, reason=f"Used {agent_output['tool_used']} instead of {expected_tool}.")
    
    # In a real system, you could ask the LLM to grade the 'response' text for tone and accuracy here.
    return AgentGrade(is_correct=True, reason="Perfect execution.")

# Run the Eval Harness
passed = 0
for test in test_cases:
    agent_result = mock_run_agent(test["input"])
    grade = grade_agent_response(test["input"], agent_result, test["expected_tool"])
    
    if grade.is_correct:
        print("✅ PASS")
        passed += 1
    else:
        print(f"❌ FAIL: {grade.reason}")

print(f"\nFinal Eval Score: {passed}/{len(test_cases)}")


⚖️ Grading ticket: 'My account is locked.'
✅ PASS
⚖️ Grading ticket: 'I need a $50 refund for my pro plan.'
❌ FAIL: Used escalate instead of prepare_refund.
⚖️ Grading ticket: 'The EU server is down.'
✅ PASS

Final Eval Score: 2/3


## Checkpoint

**1. Why is relying on "vibes" (manual spot checking) bad for agent development?**
- A) It is illegal.
- B) Agents are non-deterministic. A system prompt change might fix one edge case but silently break 5 others. Without an automated eval harness, regression is inevitable.
- C) It is too fast.
- D) It uses too many API tokens.
